# One system, RMS and EMT

In HERMESS the network model is a per-run switch. With `line_dyn=False` the
network is quasi-static: the current balance is algebraic, which is the
phasor (RMS) setting of classic stability programs. With `line_dyn=True`
every line current and bus voltage becomes a differential state, so the
network carries its own electromagnetic transients. The system file is the
same in both runs.

The case here is `3bus_genrou`: two round-rotor GENROU machines and an
impedance load on a three-bus triangle, with the line between buses 3 and 1
opening at t = 1 s. The machines run at constant mechanical power and
constant excitation, so the comparison isolates the network model.

In [ ]:
import matplotlib.pyplot as plt

import hermess

rms = hermess.extract_results(
    hermess.simulate("3bus_genrou", T_end=3.0, ts=1e-3, line_dyn=False)
)
emt = hermess.extract_results(
    hermess.simulate("3bus_genrou", T_end=3.0, ts=1e-4, line_dyn=True)
)
print(f"quasi-static: {len(rms.t)} steps, dynamic network: {len(emt.t)} steps")

Over the full window the two runs tell the same story:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(emt.t, emt.voltage_magnitude("2"), color="#215CAF", lw=0.8,
        label="dynamic network")
ax.plot(rms.t, rms.voltage_magnitude("2"), color="#B7352D", ls="--", lw=1.2,
        label="quasi-static network")
ax.axvline(1.0, color="0.6", ls=":", lw=1)
ax.set_xlabel("t [s]")
ax.set_ylabel(r"$|v_2|$ [p.u.]")
ax.legend()
fig.tight_layout()

The difference lives in the milliseconds after the event. The quasi-static
network jumps to the new algebraic solution; the dynamic network rings at
the network's own frequencies before settling onto the same envelope:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(emt.t, emt.voltage_magnitude("2"), color="#215CAF", lw=0.9,
        label="dynamic network")
ax.plot(rms.t, rms.voltage_magnitude("2"), color="#B7352D", ls="--", lw=1.4,
        label="quasi-static network")
ax.set_xlim(0.98, 1.25)
ax.set_xlabel("t [s]")
ax.set_ylabel(r"$|v_2|$ [p.u.]")
ax.legend()
fig.tight_layout()

The rotor speeds are indistinguishable between the two settings, which is
the point of the hybrid formulation. With constant mechanical power and no
governor, the machines accelerate once the lost line unloads them, and both
network models agree on that electromechanical response exactly. The
quasi-static model answers such stability questions at a fraction of the
cost (3000 versus 30000 steps here); the dynamic model is there when the
fast dynamics themselves are the question, converter controls interacting
with the network above all.

In [ ]:
g1_rms = next(d for d in rms.devices if d.unit == "SG1")
g1_emt = next(d for d in emt.devices if d.unit == "SG1")

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(emt.t, g1_emt.states["omega"], color="#215CAF", lw=0.9,
        label="dynamic network")
ax.plot(rms.t, g1_rms.states["omega"], color="#B7352D", ls="--", lw=1.2,
        label="quasi-static network")
ax.axvline(1.0, color="0.6", ls=":", lw=1)
ax.set_xlabel("t [s]")
ax.set_ylabel(r"$\omega_{\mathrm{SG1}}$ [p.u.]")
ax.legend()
fig.tight_layout()